<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 11



<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

[ваш текст]

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) исользуйтие в проекте коллекции, делегаты, события.


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [1]:

using System;
using System.Collections.Generic;

// Делегат
public delegate void CustomerMessageHandler(string message);

// Класс для передачи информации о событии
public class CustomerEventArgs : EventArgs
{
    public Customer Customer { get; set; }
    public string Message { get; set; }

    public CustomerEventArgs(Customer customer, string message)
    {
        Customer = customer;
        Message = message;
    }
}

// Класс для информации о покупке
public class PurchaseEventArgs : EventArgs
{
    public string ProductName { get; set; }
    public decimal Amount { get; set; }

    public PurchaseEventArgs(string productName, decimal amount)
    {
        ProductName = productName;
        Amount = amount;
    }
}

public class Customer
{
    public int CustomerId;
    public string Name;

    public Customer(int customerId, string name)
    {
        CustomerId = customerId;
        Name = name;
    }
}

// Базовый класс
public abstract class BaseCustomer
{
    private int _CustomerId;
    private string _CustomerName;
    private string _Email;

    protected List<Customer> customers;

    // Коллекции
    public List<string> ActivityHistory { get; set; }
    public HashSet<string> Tags { get; set; }

    // Дополнительные свойства
    public string Address { get; set; } = "Не указан";
    public bool IsActive { get; private set; } = true;
    public DateTime CreatedAt { get; private set; } = DateTime.Now;

    public int CustomerId
    {
        get { return _CustomerId; }
        set { _CustomerId = value; }
    }

    public string CustomerName
    {
        get { return _CustomerName; }
        set { _CustomerName = value; }
    }

    public string Email
    {
        get { return _Email; }
        set { _Email = value; }
    }

    // Событие
    public event EventHandler<string> CustomerUpdated;

    public BaseCustomer(int customerId, string customerName, string email)
    {
        CustomerId = customerId;
        CustomerName = customerName;
        Email = email;

        customers = new List<Customer>();
        ActivityHistory = new List<string>();
        Tags = new HashSet<string>();
    }

    public void Activate()
    {
        IsActive = true;
        AddActivity("Клиент активирован");
        Console.WriteLine($"Клиент {CustomerName} активирован.");
    }

    public void Deactivate()
    {
        IsActive = false;
        AddActivity("Клиент деактивирован");
        Console.WriteLine($"Клиент {CustomerName} деактивирован.");
    }

    public void UpdateAddress(string newAddress)
    {
        Address = newAddress;
        AddActivity("Изменен адрес");

        CustomerUpdated?.Invoke(this, $"Адрес изменен: {Address}");
    }

    public void AddActivity(string activity)
    {
        ActivityHistory.Add($"{DateTime.Now}: {activity}");
    }

    public void AddTag(string tag)
    {
        if (Tags.Add(tag))
            Console.WriteLine($"Тег '{tag}' добавлен.");
    }

    public void RemoveTag(string tag)
    {
        Tags.Remove(tag);
    }

    public void DisplayActivityHistory()
    {
        foreach (string activity in ActivityHistory)
            Console.WriteLine(activity);
    }
}

// Интерфейсы
public interface ICustomerNotification
{
    void Send(string message);
}

public class ConsoleCustomerNotification : ICustomerNotification
{
    public void Send(string message)
    {
        Console.WriteLine($"Уведомление: {message}");
    }
}

public interface IManageable
{
    void ChangeCustomer(string newName);
}

// Менеджер клиентов
public class CustomerManager : BaseCustomer
{
    private readonly ICustomerNotification _notification;

    // Коллекция Dictionary
    public Dictionary<int, Customer> CustomerDictionary { get; set; }

    // Делегат
    public CustomerMessageHandler MessageHandler;

    public DateTime LastUpdated { get; set; }
    public string Notes { get; set; } = "";
    public decimal DiscountPercent { get; private set; }
    public decimal TotalSpent { get; private set; }

    // События
    public event EventHandler<CustomerEventArgs> CustomerAdded;
    public event EventHandler<CustomerEventArgs> CustomerRemoved;

    public CustomerManager(
        int customerId,
        string customerName,
        string email,
        ICustomerNotification notification)
        : base(customerId, customerName, email)
    {
        _notification = notification;
        CustomerDictionary = new Dictionary<int, Customer>();
    }

    public virtual void DisplayCustomerInfo()
    {
        Console.WriteLine(
            $"Клиент: {CustomerName}, ID: {CustomerId}, Email: {Email}");
    }

    public virtual void AddCustomer(Customer customer)
    {
        if (CustomerDictionary.ContainsKey(customer.CustomerId))
        {
            Console.WriteLine("Клиент с таким ID уже существует.");
            return;
        }

        customers.Add(customer);
        CustomerDictionary.Add(customer.CustomerId, customer);

        LastUpdated = DateTime.Now;
        AddActivity($"Добавлен клиент {customer.Name}");

        _notification.Send($"Клиент '{customer.Name}' добавлен.");

        CustomerAdded?.Invoke(this,
            new CustomerEventArgs(customer, "Клиент добавлен"));

        MessageHandler?.Invoke($"Добавлен клиент: {customer.Name}");
    }

    public virtual void RemoveCustomer(Customer customer)
    {
        if (CustomerDictionary.Remove(customer.CustomerId))
        {
            customers.Remove(customer);

            LastUpdated = DateTime.Now;
            AddActivity($"Удален клиент {customer.Name}");

            CustomerRemoved?.Invoke(this,
                new CustomerEventArgs(customer, "Клиент удален"));

            Console.WriteLine($"Клиент '{customer.Name}' удален.");
        }
        else
        {
            Console.WriteLine($"Клиент '{customer.Name}' не найден.");
        }
    }

    public void AddNote(string note)
    {
        Notes += note + Environment.NewLine;
        Console.WriteLine("Заметка добавлена.");
    }

    public void ApplyDiscount(decimal discount)
    {
        if (discount < 0 || discount > 100)
            throw new ArgumentOutOfRangeException(nameof(discount));

        DiscountPercent = discount;
        Console.WriteLine($"Установлена скидка: {discount}%");
    }

    public void RegisterPurchase(decimal amount)
    {
        if (amount < 0)
            throw new ArgumentOutOfRangeException(nameof(amount));

        TotalSpent += amount;
        LastUpdated = DateTime.Now;
        AddActivity($"Покупка на сумму {amount}");

        Console.WriteLine($"Общая сумма покупок: {TotalSpent}");
    }

    public void DisplayAllCustomers()
    {
        foreach (var customer in CustomerDictionary)
        {
            Console.WriteLine(
                $"ID: {customer.Key}, Имя: {customer.Value.Name}");
        }
    }
}

// VIP-клиент
public class VipCustomer : CustomerManager, IManageable
{
    private string _PhoneNumber;
    private int _LoyaltyPoints;

    public string PhoneNumber
    {
        get { return _PhoneNumber; }
        set { _PhoneNumber = value; }
    }

    public int LoyaltyPoints
    {
        get { return _LoyaltyPoints; }
        set { _LoyaltyPoints = value; }
    }

    public int VipLevel { get; private set; } = 1;
    public string PersonalManager { get; set; } = "Не назначен";
    public decimal BonusMultiplier { get; set; } = 2m;

    public event EventHandler<string> BonusAdded;

    public VipCustomer(
        int customerId,
        string customerName,
        string email,
        string phoneNumber,
        ICustomerNotification notification)
        : base(customerId, customerName, email, notification)
    {
        PhoneNumber = phoneNumber;
        LoyaltyPoints = 0;
    }

    public override void DisplayCustomerInfo()
    {
        Console.WriteLine(
            $"VIP-клиент: {CustomerName}, ID: {CustomerId}, " +
            $"Email: {Email}, Телефон: {PhoneNumber}, " +
            $"Баллы: {LoyaltyPoints}, Уровень: {VipLevel}");
    }

    // Явная реализация интерфейса
    void IManageable.ChangeCustomer(string newName)
    {
        Console.WriteLine(
            $"VIP-клиент сменил имя на '{newName}'.");

        CustomerName = newName;
    }

    public void AddLoyaltyPoints(int points)
    {
        if (points < 0)
            throw new ArgumentOutOfRangeException(nameof(points));

        LoyaltyPoints += points;

        BonusAdded?.Invoke(this, $"Начислено {points} баллов.");

        Console.WriteLine($"Начислено {points} бонусных баллов.");
    }

    public void AddLoyaltyPoints(int points, string reason)
    {
        if (points < 0)
            throw new ArgumentOutOfRangeException(nameof(points));

        LoyaltyPoints += points;

        BonusAdded?.Invoke(this, $"Начислено {points} баллов.");

        Console.WriteLine(
            $"Начислено {points} баллов. Причина: {reason}");
    }

    public void GetVipBenefits()
    {
        Console.WriteLine(
            $"VIP-клиент {CustomerName} получает специальные предложения.");
    }

    public void UpgradeVipLevel()
    {
        VipLevel++;
        Console.WriteLine($"Новый VIP-уровень: {VipLevel}");
    }

    public int CalculateBonus(decimal amount)
    {
        return (int)(amount * BonusMultiplier / 100);
    }
}

// Обычный клиент
public class RegularCustomer : CustomerManager, IManageable
{
    private DateTime _RegistrationDate;
    private int _PurchaseCount;

    public DateTime RegistrationDate
    {
        get { return _RegistrationDate; }
        set { _RegistrationDate = value; }
    }

    public int PurchaseCount
    {
        get { return _PurchaseCount; }
        set { _PurchaseCount = value; }
    }

    public DateTime LastPurchaseDate { get; private set; }
    public int TotalOrders { get; private set; }
    public bool IsSubscribed { get; private set; }

    public event EventHandler<PurchaseEventArgs> PurchaseMade;

    public RegularCustomer(
        int customerId,
        string customerName,
        string email,
        ICustomerNotification notification)
        : base(customerId, customerName, email, notification)
    {
        RegistrationDate = DateTime.Now;
        PurchaseCount = 0;
    }

    public override void DisplayCustomerInfo()
    {
        Console.WriteLine(
            $"Обычный клиент: {CustomerName}, ID: {CustomerId}, " +
            $"Email: {Email}, Покупок: {PurchaseCount}, " +
            $"Дата регистрации: {RegistrationDate}");
    }

    // Явная реализация интерфейса
    void IManageable.ChangeCustomer(string newName)
    {
        Console.WriteLine(
            $"Имя клиента изменено на '{newName}'.");

        CustomerName = newName;
    }

    public void MakePurchase()
    {
        MakePurchase("Неизвестный товар", 0);
    }

    public void MakePurchase(string productName)
    {
        MakePurchase(productName, 0);
    }

    public void MakePurchase(string productName, decimal amount)
    {
        PurchaseCount++;
        TotalOrders++;
        LastPurchaseDate = DateTime.Now;

        RegisterPurchase(amount);

        Console.WriteLine(
            $"Клиент {CustomerName} купил товар '{productName}'.");

        PurchaseMade?.Invoke(this,
            new PurchaseEventArgs(productName, amount));
    }

    public void DisplayRegistrationInfo()
    {
        Console.WriteLine(
            $"Клиент {CustomerName} зарегистрирован: {RegistrationDate}");
    }

    public void Subscribe()
    {
        IsSubscribed = true;
        Console.WriteLine("Клиент подписался на рассылку.");
    }

    public void Unsubscribe()
    {
        IsSubscribed = false;
        Console.WriteLine("Клиент отписался от рассылки.");
    }

    public double GetAveragePurchaseCount()
    {
        return TotalOrders;
    }
}

// Групповой клиент
public class GroupCustomer : CustomerManager
{
    private string _GroupName;
    private int _GroupSize;

    public string GroupName
    {
        get { return _GroupName; }
        set { _GroupName = value; }
    }

    public int GroupSize
    {
        get { return _GroupSize; }
        set { _GroupSize = value; }
    }

    public decimal GroupDiscount { get; private set; }
    public string GroupDescription { get; set; } = "";
    public bool IsClosed { get; private set; }

    public event EventHandler<string> GroupChanged;

    public GroupCustomer(
        int customerId,
        string customerName,
        string email,
        string groupName,
        ICustomerNotification notification)
        : base(customerId, customerName, email, notification)
    {
        GroupName = groupName;
        GroupSize = 0;
    }

    public override void DisplayCustomerInfo()
    {
        Console.WriteLine(
            $"Групповой клиент: {CustomerName}, ID: {CustomerId}, " +
            $"Email: {Email}, Группа: {GroupName}, " +
            $"Участников: {GroupSize}");
    }

    public override void AddCustomer(Customer customer)
    {
        if (IsClosed)
        {
            Console.WriteLine("Группа закрыта.");
            return;
        }

        int oldCount = GroupSize;

        base.AddCustomer(customer);

        if (GroupSize == oldCount &&
            CustomerDictionary.ContainsKey(customer.CustomerId))
        {
            GroupSize++;
            GroupChanged?.Invoke(this,
                $"Добавлен участник: {customer.Name}");
        }
    }

    public void AddGroupMember(Customer customer)
    {
        AddCustomer(customer);
    }

    public void DisplayGroupInfo()
    {
        Console.WriteLine(
            $"Группа: {GroupName}, участников: {GroupSize}");
    }

    public void ChangeGroupName(string newGroupName)
    {
        GroupName = newGroupName;

        GroupChanged?.Invoke(this,
            $"Название группы изменено на {newGroupName}");

        Console.WriteLine($"Новое название группы: {GroupName}");
    }

    public void ChangeGroupDiscount(decimal discount)
    {
        if (discount < 0 || discount > 100)
            throw new ArgumentOutOfRangeException(nameof(discount));

        GroupDiscount = discount;
        Console.WriteLine($"Скидка группы: {GroupDiscount}%");
    }

    public void CloseGroup()
    {
        IsClosed = true;
        GroupChanged?.Invoke(this, "Группа закрыта");
        Console.WriteLine($"Группа '{GroupName}' закрыта.");
    }

    public void OpenGroup()
    {
        IsClosed = false;
        GroupChanged?.Invoke(this, "Группа открыта");
        Console.WriteLine($"Группа '{GroupName}' открыта.");
    }
}

// ========================================
// СОЗДАНИЕ ОБЪЕКТОВ И ПРОВЕРКА ПРОГРАММЫ
// ========================================

ICustomerNotification notification =
    new ConsoleCustomerNotification();

CustomerManager myCustomerManager =
    new CustomerManager(
        1, "Анна Иванова", "anna@mail.com", notification);

VipCustomer myVipCustomer =
    new VipCustomer(
        2, "Алексей Петров", "alex@mail.com",
        "+79990000001", notification);

RegularCustomer myRegularCustomer =
    new RegularCustomer(
        3, "Мария Смирнова", "maria@mail.com", notification);

GroupCustomer myGroupCustomer =
    new GroupCustomer(
        4, "Иван Сидоров", "ivan@mail.com",
        "Постоянные клиенты", notification);

// Подписка на события
myCustomerManager.CustomerAdded += (sender, e) =>
    Console.WriteLine($"Событие: {e.Message} — {e.Customer.Name}");

myCustomerManager.CustomerRemoved += (sender, e) =>
    Console.WriteLine($"Событие: {e.Message} — {e.Customer.Name}");

myCustomerManager.CustomerUpdated += (sender, message) =>
    Console.WriteLine($"Событие изменения: {message}");

myVipCustomer.BonusAdded += (sender, message) =>
    Console.WriteLine($"Событие VIP: {message}");

myRegularCustomer.PurchaseMade += (sender, e) =>
    Console.WriteLine(
        $"Событие покупки: {e.ProductName}, сумма: {e.Amount}");

myGroupCustomer.GroupChanged += (sender, message) =>
    Console.WriteLine($"Событие группы: {message}");

// Подписка на делегат
myCustomerManager.MessageHandler = message =>
    Console.WriteLine($"Делегат: {message}");

// Работа с VIP-клиентом
Console.WriteLine("=== VIP-КЛИЕНТ ===");

myVipCustomer.DisplayCustomerInfo();
myVipCustomer.AddLoyaltyPoints(100);
myVipCustomer.AddLoyaltyPoints(50, "Покупка на день рождения");
myVipCustomer.UpgradeVipLevel();
myVipCustomer.GetVipBenefits();

IManageable vipManager = myVipCustomer;
vipManager.ChangeCustomer("Алексей Петрович");

myVipCustomer.AddTag("VIP");
myVipCustomer.AddTag("Постоянный клиент");

Console.WriteLine();

// Работа с обычным клиентом
Console.WriteLine("=== ОБЫЧНЫЙ КЛИЕНТ ===");

myRegularCustomer.DisplayCustomerInfo();
myRegularCustomer.MakePurchase("Ноутбук", 50000);
myRegularCustomer.MakePurchase("Мышь", 2000);
myRegularCustomer.Subscribe();

IManageable regularManager = myRegularCustomer;
regularManager.ChangeCustomer("Мария Иванова");

Console.WriteLine();

// Работа с группой
Console.WriteLine("=== ГРУППОВОЙ КЛИЕНТ ===");

myGroupCustomer.AddGroupMember(
    new Customer(5, "Елена Кузнецова"));

myGroupCustomer.AddGroupMember(
    new Customer(6, "Дмитрий Волков"));

myGroupCustomer.DisplayGroupInfo();
myGroupCustomer.ChangeGroupName("VIP-группа");
myGroupCustomer.ChangeGroupDiscount(10);

Console.WriteLine();

// Работа с коллекциями
Console.WriteLine("=== КОЛЛЕКЦИИ ===");

myCustomerManager.AddCustomer(
    new Customer(10, "Иван Петров"));

myCustomerManager.AddCustomer(
    new Customer(11, "Ольга Соколова"));

myCustomerManager.DisplayAllCustomers();

Console.WriteLine();

// Удаление клиента
myCustomerManager.RemoveCustomer(
    new Customer(10, "Иван Петров"));

Console.WriteLine();

// Работа с базовым классом
Console.WriteLine("=== БАЗОВЫЙ КЛАСС ===");

myVipCustomer.UpdateAddress("г. Тюмень, ул. Республики");
myVipCustomer.AddActivity("Посещение магазина");
myVipCustomer.DisplayActivityHistory();

Console.WriteLine();

// Работа с полиморфизмом
Console.WriteLine("=== ПОЛИМОРФИЗМ ===");

CustomerManager[] customerList =
{
    myCustomerManager,
    myVipCustomer,
    myRegularCustomer,
    myGroupCustomer
};

foreach (CustomerManager customer in customerList)
{
    customer.DisplayCustomerInfo();
    Console.WriteLine();
}


=== VIP-КЛИЕНТ ===
VIP-клиент: Алексей Петров, ID: 2, Email: alex@mail.com, Телефон: +79990000001, Баллы: 0, Уровень: 1
Событие VIP: Начислено 100 баллов.
Начислено 100 бонусных баллов.
Событие VIP: Начислено 50 баллов.
Начислено 50 баллов. Причина: Покупка на день рождения
Новый VIP-уровень: 2
VIP-клиент Алексей Петров получает специальные предложения.
VIP-клиент сменил имя на 'Алексей Петрович'.
Тег 'VIP' добавлен.
Тег 'Постоянный клиент' добавлен.

=== ОБЫЧНЫЙ КЛИЕНТ ===
Обычный клиент: Мария Смирнова, ID: 3, Email: maria@mail.com, Покупок: 0, Дата регистрации: 11/17/2026 6:37:28 PM
Общая сумма покупок: 50000
Клиент Мария Смирнова купил товар 'Ноутбук'.
Событие покупки: Ноутбук, сумма: 50000
Общая сумма покупок: 52000
Клиент Мария Смирнова купил товар 'Мышь'.
Событие покупки: Мышь, сумма: 2000
Клиент подписался на рассылку.
Имя клиента изменено на 'Мария Иванова'.

=== ГРУППОВОЙ КЛИЕНТ ===
Уведомление: Клиент 'Елена Кузнецова' добавлен.
Событие группы: Добавлен участник: Елена Кузне